# MLOps Example Notebook - Iris Classifier v2.1

This notebook demonstrates the required cell tags for the MLOps platform.
Each code cell has a `tags` entry in its metadata that the platform uses
to identify pipeline phases.

**Required tags:** `mlops:config`, `mlops:preprocessing`, `mlops:training`, `mlops:export`

**Optional tags:** `mlops:data`, `mlops:evaluation`

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.
MODEL_OUTPUT_PATH = "./model.joblib"
PIPELINE_ID = "local-dev"
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_RUN_ID = "local-dev"

In [ ]:
# ============================================================
# mlops:config - Model metadata
# ============================================================
# Adapt MODEL_NAME and VERSION for your project.
# The platform reads these values to register the model.

MODEL_NAME = "iris-classifier"
VERSION = "1"

print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")

In [ ]:
# --- Cell 3 [tags: mlops:data] ---
# ============================================================
# mlops:data - Data loading
# ============================================================
# Replace this with your own data loading logic.
# The platform does not require a specific data source.

import os
from sklearn.datasets import load_iris
import pandas as pd
from sklearn.preprocessing import LabelEncoder # Needed if target_column_name is not numerical

# Verifica si DATASET_PATH ha sido inyectado por la plataforma
DATASET_PATH = os.getenv("DATASET_PATH")
CLASS_NAMES = [] # Initialize CLASS_NAMES for later use

if DATASET_PATH and os.path.exists(DATASET_PATH):
    print(f"Cargando dataset desde: {DATASET_PATH}")
    # Asume que el dataset es un CSV; ajusta según tu formato
    try:
        df = pd.read_csv(DATASET_PATH)
        # Asegúrate de ajustar las columnas y la columna objetivo según tu dataset
        # *** IMPORTANTE: Reemplaza 'target_column_name' con el nombre real de tu columna objetivo ***
        if 'target_column_name' in df.columns:
            X = df.drop(columns=['target_column_name'])
            y = df['target_column_name']
        else:
            # Fallback heuristic: assume the last column is the target.
            # In a real scenario, this should be explicitly configured.
            print("Advertencia: 'target_column_name' no encontrada. Asumiendo que la última columna es la columna objetivo.")
            X = df.iloc[:, :-1]
            y = df.iloc[:, -1]

        # Si target_column_name no es numérica, es posible que necesites codificarla
        if y.dtype == 'object' or y.dtype == 'category':
            print("Codificando columna objetivo usando LabelEncoder...")
            le = LabelEncoder()
            y = le.fit_transform(y)
            CLASS_NAMES = le.classes_.tolist() # Store original class names
        else:
            # If numerical, just get unique classes as strings for consistency
            CLASS_NAMES = [str(c) for c in sorted(y.unique())]

    except Exception as e:
        print(f"Error cargando el dataset desde {DATASET_PATH}: {e}. Usando dataset Iris por defecto.")
        iris = load_iris()
        X = pd.DataFrame(iris.data, columns=iris.feature_names)
        y = pd.Series(iris.target, name="target")
        CLASS_NAMES = iris.target_names.tolist() # Store Iris class names
else:
    print("DATASET_PATH no inyectado o no encontrado. Usando dataset Iris por defecto.")
    iris = load_iris()
    X = pd.DataFrame(iris.data, columns=iris.feature_names)
    y = pd.Series(iris.target, name="target")
    CLASS_NAMES = iris.target_names.tolist() # Store Iris class names

print(f"Dataset shape: {X.shape}")
# Asegúrate de que `CLASS_NAMES` sea compatible o reemplázalo con tus propios nombres de clase
if CLASS_NAMES:
    print(f"Classes: {list(CLASS_NAMES)}")
else:
    print(f"Clases únicas en y: {y.nunique()}") # Fallback if CLASS_NAMES is empty (shouldn't happen with the current logic)


In [ ]:
# ============================================================
# mlops:preprocessing - Data preparation
# ============================================================
# Add feature engineering, scaling, splitting, etc.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

In [ ]:
# --- Cell 5 [tags: mlops:training] ---
# ============================================================
# mlops:training - Model training
# ============================================================
# Replace RandomForestClassifier with your own model.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import mlflow # Importado para loguear los parámetros si fuera necesario, aunque se usa principalmente en la celda de evaluación.

# Definir el espacio de búsqueda de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
}

# Inicializar el modelo base
rf = RandomForestClassifier(random_state=42)

# Configurar GridSearchCV
# n_jobs=-1 para usar todos los procesadores disponibles
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Entrenar el modelo con búsqueda de hiperparámetros
grid_search.fit(X_train_scaled, y_train)

# Seleccionar el mejor modelo
model = grid_search.best_estimator_

print("Training complete.")
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor score de CV: {grid_search.best_score_:.4f}")


In [ ]:
# --- Cell 6 [tags: mlops:evaluation] ---
# ============================================================
# mlops:evaluation - Metrics and logging
# ============================================================
# Log metrics to MLflow. The platform reads 'accuracy' for
# auto-deployment decisions.

from sklearn.metrics import accuracy_score, classification_report
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

y_pred = model.predict(X_test_scaled)
score = accuracy_score(y_test, y_pred)

# Usar CLASS_NAMES para target_names que se popula en la Celda 3
print(classification_report(y_test, y_pred, target_names=list(CLASS_NAMES)))
print(f"Accuracy: {score:.4f}")

# Use the run started by the pipeline if available, otherwise start a new one
run_ctx = mlflow.start_run(run_id=MLFLOW_RUN_ID) if MLFLOW_RUN_ID != "local-dev" else mlflow.start_run()
with run_ctx:
    mlflow.log_metric("accuracy", score)
    # Loguea todos los parámetros del modelo entrenado (el mejor de GridSearchCV)
    mlflow.log_params(model.get_params())
    # Loguea el mejor score de CV encontrado por GridSearchCV
    mlflow.log_metric("best_cv_score", grid_search.best_score_)


In [ ]:
# ============================================================
# mlops:export - Save model artifact
# ============================================================
# Save the trained model to MODEL_OUTPUT_PATH.
# The platform picks up this file and registers it in MLflow.

import joblib

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to {MODEL_OUTPUT_PATH}")